In [2]:
import hl7
import fhir.resources
import requests
import pandas
print("All libraries loaded successfully")

All libraries loaded successfully


In [3]:
adt_message = """MSH|^~\\&|MS4|HOSP|RECEIVER|RECEIVER|20260101120000||ADT^A01|MSG00001|P|2.5
EVN|A01|20260101120000
PID|1||1234567^^^HOSP^MR||DOE^JANE||19850315|F|||123 MAIN ST^^SANTA CLARA^CA^95050||4085551234
PV1|1|I|WARD1^101^A|||ATTEND123^SMITH^JOHN|||MED||||||||INS1|A0"""

oru_message = """MSH|^~\\&|SIS|HOSP|RECEIVER|RECEIVER|20260101130000||ORU^R01|MSG00002|P|2.5
PID|1||1234567^^^HOSP^MR||DOE^JANE||19850315|F
OBR|1|ORD001|RES001|CBC^Complete Blood Count||20260101120500
OBX|1|NM|718-7^Hemoglobin^LN||13.5|g/dL|12.0-15.5|N|||F"""

print("ADT message ready:")
print(adt_message)
print()
print("ORU message ready:")
print(oru_message)

ADT message ready:
MSH|^~\&|MS4|HOSP|RECEIVER|RECEIVER|20260101120000||ADT^A01|MSG00001|P|2.5
EVN|A01|20260101120000
PID|1||1234567^^^HOSP^MR||DOE^JANE||19850315|F|||123 MAIN ST^^SANTA CLARA^CA^95050||4085551234
PV1|1|I|WARD1^101^A|||ATTEND123^SMITH^JOHN|||MED||||||||INS1|A0

ORU message ready:
MSH|^~\&|SIS|HOSP|RECEIVER|RECEIVER|20260101130000||ORU^R01|MSG00002|P|2.5
PID|1||1234567^^^HOSP^MR||DOE^JANE||19850315|F
OBR|1|ORD001|RES001|CBC^Complete Blood Count||20260101120500
OBX|1|NM|718-7^Hemoglobin^LN||13.5|g/dL|12.0-15.5|N|||F


In [4]:
import hl7

# hl7 library expects \r (carriage return) between segments, not \n
adt_parsed = hl7.parse(adt_message.replace("\n", "\r"))
oru_parsed = hl7.parse(oru_message.replace("\n", "\r"))

# Pull the PID segment out of the ADT message
pid_segment = adt_parsed.segment("PID")

family_name = str(pid_segment[5][0][0])
given_name = str(pid_segment[5][0][1])
dob_raw = str(pid_segment[7][0])          # format: YYYYMMDD
gender_code = str(pid_segment[8][0])

print("Family name:", family_name)
print("Given name:", given_name)
print("DOB (raw):", dob_raw)
print("Gender code:", gender_code)

Family name: DOE
Given name: JANE
DOB (raw): 19850315
Gender code: F


In [5]:
obx_segment = oru_parsed.segment("OBX")

loinc_code = str(obx_segment[3][0][0])
loinc_display = str(obx_segment[3][0][1])
lab_value = str(obx_segment[5][0])
lab_unit = str(obx_segment[6][0])

print("LOINC code:", loinc_code)
print("LOINC display:", loinc_display)
print("Lab value:", lab_value)
print("Lab unit:", lab_unit)

LOINC code: 718-7
LOINC display: Hemoglobin
Lab value: 13.5
Lab unit: g/dL


In [6]:
import uuid
from fhir.resources.patient import Patient
from fhir.resources.encounter import Encounter
from fhir.resources.observation import Observation

# Generate unique IDs we'll reuse to link resources together
patient_id = str(uuid.uuid4())
encounter_id = str(uuid.uuid4())
observation_id = str(uuid.uuid4())

# Reformat DOB from HL7's YYYYMMDD into FHIR's required YYYY-MM-DD
dob_formatted = f"{dob_raw[0:4]}-{dob_raw[4:6]}-{dob_raw[6:8]}"

# Map HL7's single-letter gender code to FHIR's required word format
gender_map = {"M": "male", "F": "female", "O": "other", "U": "unknown"}
gender_formatted = gender_map.get(gender_code, "unknown")

# ---- Build Patient as a FHIR-shaped dictionary ----
patient_dict = {
    "resourceType": "Patient",
    "id": patient_id,
    "identifier": [{"system": "urn:hl7-mrn", "value": "1234567"}],
    "name": [{"family": family_name, "given": [given_name]}],
    "gender": gender_formatted,
    "birthDate": dob_formatted
}

patient_resource = Patient.model_validate(patient_dict)
print("Patient resource is valid FHIR. ID:", patient_resource.id)

Patient resource is valid FHIR. ID: 0dcd6863-c9e5-43d8-8eaa-df854e4dd964


In [7]:
encounter_dict = {
    "resourceType": "Encounter",
    "id": encounter_id,
    "status": "in-progress",
    "class": [{
    "coding": [{
        "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
        "code": "IMP",
        "display": "inpatient encounter"
    }]
}],
    "subject": {"reference": f"urn:uuid:{patient_id}"}
}

encounter_resource = Encounter.model_validate(encounter_dict)
print("Encounter resource is valid FHIR. ID:", encounter_resource.id)

Encounter resource is valid FHIR. ID: cb514759-93f2-43ac-83b2-aaa1216cdd49


In [8]:
observation_dict = {
    "resourceType": "Observation",
    "id": observation_id,
    "status": "final",
    "code": {
        "coding": [{
            "system": "http://loinc.org",
            "code": loinc_code,
            "display": loinc_display
        }]
    },
    "subject": {"reference": f"urn:uuid:{patient_id}"},
    "encounter": {"reference": f"urn:uuid:{encounter_id}"},
    "valueQuantity": {
        "value": float(lab_value),
        "unit": lab_unit,
        "system": "http://unitsofmeasure.org",
        "code": lab_unit
    }
}

observation_resource = Observation.model_validate(observation_dict)
print("Observation resource is valid FHIR. ID:", observation_resource.id)

Observation resource is valid FHIR. ID: b3cccbba-9cd6-45fd-96dc-8f1edcda2d9f


In [9]:
from fhir.resources.bundle import Bundle

bundle_dict = {
    "resourceType": "Bundle",
    "type": "transaction",
    "entry": [
        {
            "fullUrl": f"urn:uuid:{patient_id}",
            "resource": patient_dict,
            "request": {"method": "POST", "url": "Patient"}
        },
        {
            "fullUrl": f"urn:uuid:{encounter_id}",
            "resource": encounter_dict,
            "request": {"method": "POST", "url": "Encounter"}
        },
        {
            "fullUrl": f"urn:uuid:{observation_id}",
            "resource": observation_dict,
            "request": {"method": "POST", "url": "Observation"}
        }
    ]
}

bundle_resource = Bundle.model_validate(bundle_dict)
print("Bundle is valid FHIR. Type:", bundle_resource.type)
print("Number of entries:", len(bundle_resource.entry))

Bundle is valid FHIR. Type: transaction
Number of entries: 3


In [10]:
import json

with open("output_bundle.json", "w") as f:
    json.dump(bundle_dict, f, indent=2)

print("Bundle saved to output_bundle.json")

Bundle saved to output_bundle.json


In [11]:
with open("output_bundle.json", "r") as f:
    bundle_text = f.read()

print(bundle_text)

{
  "resourceType": "Bundle",
  "type": "transaction",
  "entry": [
    {
      "fullUrl": "urn:uuid:0dcd6863-c9e5-43d8-8eaa-df854e4dd964",
      "resource": {
        "resourceType": "Patient",
        "id": "0dcd6863-c9e5-43d8-8eaa-df854e4dd964",
        "identifier": [
          {
            "system": "urn:hl7-mrn",
            "value": "1234567"
          }
        ],
        "name": [
          {
            "family": "DOE",
            "given": [
              "JANE"
            ]
          }
        ],
        "gender": "female",
        "birthDate": "1985-03-15"
      },
      "request": {
        "method": "POST",
        "url": "Patient"
      }
    },
    {
      "fullUrl": "urn:uuid:cb514759-93f2-43ac-83b2-aaa1216cdd49",
      "resource": {
        "resourceType": "Encounter",
        "id": "cb514759-93f2-43ac-83b2-aaa1216cdd49",
        "status": "in-progress",
        "class": [
          {
            "coding": [
              {
                "system": "http://termino

In [12]:
import requests
import time

server_url = "https://hapi.fhir.org/baseR4"

start_time = time.time()

try:
    response = requests.post(
        server_url,
        json=bundle_dict,
        headers={"Content-Type": "application/fhir+json"},
        timeout=30
    )
    latency_seconds = time.time() - start_time
    print("HTTP Status Code:", response.status_code)
    print("Latency (seconds):", round(latency_seconds, 2))
except requests.exceptions.Timeout:
    print("Request timed out after 30 seconds. The public test server may be slow or unresponsive right now.")

HTTP Status Code: 200
Latency (seconds): 1.95


In [13]:
result = response.json()

for entry in result.get("entry", []):
    status = entry.get("response", {}).get("status", "unknown")
    location = entry.get("response", {}).get("location", "no location returned")
    print(f"Status: {status} | Location: {location}")

Status: 201 Created | Location: Patient/25784/_history/1
Status: 201 Created | Location: Encounter/25785/_history/1
Status: 201 Created | Location: Observation/25786/_history/1


In [14]:
import json

with open("server_response.json", "w") as f:
    json.dump(result, f, indent=2)

print("Server response saved.")

Server response saved.


In [15]:
get_url = "https://hapi.fhir.org/baseR4/Patient/25784"

get_response = requests.get(get_url, timeout=30)

print("GET Status Code:", get_response.status_code)
print(json.dumps(get_response.json(), indent=2))

GET Status Code: 200
{
  "resourceType": "Patient",
  "id": "25784",
  "meta": {
    "versionId": "1",
    "lastUpdated": "2026-09-17T20:46:53.145-04:00",
    "source": "#g6zRkpgDiNCVXIkD"
  },
  "text": {
    "status": "generated",
    "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><div class=\"hapiHeaderText\">JANE <b>DOE </b></div><table class=\"hapiPropertyTable\"><tbody><tr><td>Identifier</td><td>1234567</td></tr><tr><td>Date of birth</td><td><span>15 March 1985</span></td></tr></tbody></table></div>"
  },
  "identifier": [
    {
      "system": "urn:hl7-mrn",
      "value": "1234567"
    }
  ],
  "name": [
    {
      "family": "DOE",
      "given": [
        "JANE"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "1985-03-15"
}


In [16]:
import pandas as pd
import datetime

log_entry = {
    "timestamp": datetime.datetime.now().isoformat(),
    "message_type": "ADT_A01 + ORU_R01",
    "validation_status": "Pass (warnings only)",
    "error_count": 0,
    "http_status": response.status_code,
    "latency_seconds": round(latency_seconds, 2)
}

log_df = pd.DataFrame([log_entry])

try:
    existing_log = pd.read_csv("run_log.csv")
    log_df = pd.concat([existing_log, log_df], ignore_index=True)
except FileNotFoundError:
    pass

log_df.to_csv("run_log.csv", index=False)
print("Logged. Current run_log.csv:")
print(log_df)

Logged. Current run_log.csv:
                    timestamp       message_type     validation_status  \
0  2026-09-17T17:50:20.414114  ADT_A01 + ORU_R01  Pass (warnings only)   

   error_count  http_status  latency_seconds  
0            0          200             1.95  


In [17]:
broken_messages = [
    {
        "name": "Missing DOB",
        "adt": """MSH|^~\\&|MS4|HOSP|RECEIVER|RECEIVER|20260102120000||ADT^A01|MSG00003|P|2.5
EVN|A01|20260102120000
PID|1||2234567^^^HOSP^MR||SMITH^JOHN|||M|||456 OAK ST^^SANTA CLARA^CA^95050||4085559999
PV1|1|I|WARD1^102^A|||ATTEND124^JONES^MARY|||MED||||||||INS1|A0""",
        "expected_issue": "Missing birthDate field (PID-7 blank) - FHIR Patient.birthDate will fail to populate"
    },
    {
        "name": "Invalid gender code",
        "adt": """MSH|^~\\&|MS4|HOSP|RECEIVER|RECEIVER|20260103120000||ADT^A01|MSG00004|P|2.5
EVN|A01|20260103120000
PID|1||3234567^^^HOSP^MR||BROWN^ALEX||19900622|X|||789 PINE ST^^SANTA CLARA^CA^95050||4085558888
PV1|1|I|WARD1^103^A|||ATTEND125^LEE^SUSAN|||MED||||||||INS1|A0""",
        "expected_issue": "Gender code 'X' not in HL7 v2 table 0001 - will map to 'unknown' in FHIR, flagged as a data quality issue"
    },
    {
        "name": "Malformed lab value",
        "adt": """MSH|^~\\&|SIS|HOSP|RECEIVER|RECEIVER|20260104130000||ORU^R01|MSG00005|P|2.5
PID|1||4234567^^^HOSP^MR||GREEN^PAT||19750812|F
OBR|1|ORD002|RES002|CBC^Complete Blood Count||20260104120500
OBX|1|NM|718-7^Hemoglobin^LN||NOT_A_NUMBER|g/dL|12.0-15.5|N|||F""",
        "expected_issue": "Lab value 'NOT_A_NUMBER' cannot convert to float - will raise a Python exception during transformation"
    }
]

print(f"{len(broken_messages)} broken test cases ready")
for msg in broken_messages:
    print("-", msg["name"], ":", msg["expected_issue"])

3 broken test cases ready
- Missing DOB : Missing birthDate field (PID-7 blank) - FHIR Patient.birthDate will fail to populate
- Invalid gender code : Gender code 'X' not in HL7 v2 table 0001 - will map to 'unknown' in FHIR, flagged as a data quality issue
- Malformed lab value : Lab value 'NOT_A_NUMBER' cannot convert to float - will raise a Python exception during transformation


In [18]:
import hl7
import uuid
import datetime

def process_message(name, raw_message, message_kind):
    """Runs one message through parse -> build Patient -> log outcome"""
    log_row = {
        "timestamp": datetime.datetime.now().isoformat(),
        "message_type": name,
        "validation_status": "Unknown",
        "error_count": 0,
        "http_status": None,
        "latency_seconds": None
    }

    try:
        parsed = hl7.parse(raw_message.replace("\n", "\r"))
        pid_seg = parsed.segment("PID")

        family = str(pid_seg[5][0][0])
        given = str(pid_seg[5][0][1])
        dob_raw = str(pid_seg[7][0]) if str(pid_seg[7][0]) else ""
        gender_code = str(pid_seg[8][0])

        # This will fail loudly if DOB is missing/malformed - that's intentional
        dob_formatted = f"{dob_raw[0:4]}-{dob_raw[4:6]}-{dob_raw[6:8]}"

        gender_map = {"M": "male", "F": "female", "O": "other", "U": "unknown"}
        gender_formatted = gender_map.get(gender_code, "unknown")

        test_patient = {
            "resourceType": "Patient",
            "id": str(uuid.uuid4()),
            "name": [{"family": family, "given": [given]}],
            "gender": gender_formatted,
            "birthDate": dob_formatted
        }

        from fhir.resources.patient import Patient
        Patient.model_validate(test_patient)

        log_row["validation_status"] = "Pass"
        print(f"[{name}] PASSED - built valid Patient resource")

    except Exception as e:
        log_row["validation_status"] = "Fail"
        log_row["error_count"] = 1
        print(f"[{name}] FAILED - {type(e).__name__}: {e}")

    return log_row

# Run all three broken cases
new_rows = []
for case in broken_messages:
    row = process_message(case["name"], case["adt"], "ADT_A01")
    new_rows.append(row)

[Missing DOB] FAILED - ValidationError: 1 validation error for Patient
birthDate
  Value error, Date value string does not match spec regex. [type=value_error, input_value='--', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
[Invalid gender code] PASSED - built valid Patient resource
[Malformed lab value] PASSED - built valid Patient resource


In [19]:
import pandas as pd

new_log_df = pd.DataFrame(new_rows)
existing_log = pd.read_csv("run_log.csv")
combined_log = pd.concat([existing_log, new_log_df], ignore_index=True)
combined_log.to_csv("run_log.csv", index=False)

print("Updated run_log.csv:")
print(combined_log)

Updated run_log.csv:
                    timestamp         message_type     validation_status  \
0  2026-09-17T17:50:20.414114    ADT_A01 + ORU_R01  Pass (warnings only)   
1  2026-09-17T17:56:12.682496          Missing DOB                  Fail   
2  2026-09-17T17:56:12.682496  Invalid gender code                  Pass   
3  2026-09-17T17:56:12.682496  Malformed lab value                  Pass   

   error_count http_status latency_seconds  
0            0         200            1.95  
1            1        None            None  
2            0        None            None  
3            0        None            None  


In [20]:
def process_lab_message(name, raw_message):
    """Runs an ORU message through parse -> build Observation -> log outcome"""
    log_row = {
        "timestamp": datetime.datetime.now().isoformat(),
        "message_type": name,
        "validation_status": "Unknown",
        "error_count": 0,
        "http_status": None,
        "latency_seconds": None
    }

    try:
        parsed = hl7.parse(raw_message.replace("\n", "\r"))
        obx_seg = parsed.segment("OBX")

        loinc_code = str(obx_seg[3][0][0])
        loinc_display = str(obx_seg[3][0][1])
        lab_value_raw = str(obx_seg[5][0])
        lab_unit = str(obx_seg[6][0])

        lab_value_numeric = float(lab_value_raw)

        test_observation = {
            "resourceType": "Observation",
            "id": str(uuid.uuid4()),
            "status": "final",
            "code": {"coding": [{"system": "http://loinc.org", "code": loinc_code, "display": loinc_display}]},
            "valueQuantity": {"value": lab_value_numeric, "unit": lab_unit}
        }

        from fhir.resources.observation import Observation
        Observation.model_validate(test_observation)

        log_row["validation_status"] = "Pass"
        print(f"[{name}] PASSED - built valid Observation resource")

    except Exception as e:
        log_row["validation_status"] = "Fail"
        log_row["error_count"] = 1
        print(f"[{name}] FAILED - {type(e).__name__}: {e}")

    return log_row

lab_case = broken_messages[2]
corrected_row = process_lab_message(lab_case["name"], lab_case["adt"])

[Malformed lab value] FAILED - ValueError: could not convert string to float: 'NOT_A_NUMBER'


In [21]:
combined_log = pd.read_csv("run_log.csv")
combined_log = combined_log[combined_log["message_type"] != "Malformed lab value"]

corrected_df = pd.DataFrame([corrected_row])
combined_log = pd.concat([combined_log, corrected_df], ignore_index=True)
combined_log.to_csv("run_log.csv", index=False)

print("Corrected run_log.csv:")
print(combined_log)

Corrected run_log.csv:
                    timestamp         message_type     validation_status  \
0  2026-09-17T17:50:20.414114    ADT_A01 + ORU_R01  Pass (warnings only)   
1  2026-09-17T17:56:12.682496          Missing DOB                  Fail   
2  2026-09-17T17:56:12.682496  Invalid gender code                  Pass   
3  2026-09-21T09:51:23.618573  Malformed lab value                  Fail   

   error_count http_status latency_seconds  
0            0       200.0            1.95  
1            1         NaN             NaN  
2            0         NaN             NaN  
3            1        None            None  
